In [ ]:
# Import required libraries
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import torch

In [ ]:
#Change between "MLRS/BERTu" and "MLRS/mBERTu" to swap transformer
model_name = "MLRS/BERTu"

# Different Model Configurations
output_dir = "./multi_model_runs"
os.makedirs(output_dir, exist_ok = True)

model_runs = [
    {
        "run_name": "1_lr2e5_bs16_wd001",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "num_train_epochs": 200,
        "weight_decay": 0.01,
        "max_length": 128,
    },
    {
        "run_name": "2_lr3e5_bs16_wd001",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "num_train_epochs": 200,
        "weight_decay": 0.01,
        "max_length": 128,
    },    
    {
        "run_name": "3_lr1e5_bs16_wd001",
        "learning_rate": 1e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "num_train_epochs": 200,
        "weight_decay": 0.01,
        "max_length": 128,
    },     
    {
        "run_name": "4_lr2e5_bs16_wd005",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "num_train_epochs": 200,
        "weight_decay": 0.05,
        "max_length": 128,
    },
    {
        "run_name": "5_lr2e5_bs16_wd01",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "num_train_epochs": 200,
        "weight_decay": 0.1,
        "max_length": 128,
    },
    {
        "run_name": "6_lr2e5_bs32_wd001",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": 200,
        "weight_decay": 0.01,
        "max_length": 128,
    },
    {
        "run_name": "7_lr3e5_bs32_wd001",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": 200,
        "weight_decay": 0.01,
        "max_length": 128,
    },
    {
        "run_name": "8_lr2e5_bs32_wd005",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": 200,
        "weight_decay": 0.05,
        "max_length": 128,
    },
    {
        "run_name": "9_lr3e5_bs32_wd005",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": 200,
        "weight_decay": 0.05,
        "max_length": 128,
    },
        {
        "run_name": "10_lr3e5_bs32_wd01",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": 200,
        "weight_decay": 0.1,
        "max_length": 128,
    },
]

# Mapping the dataset results
label_map = {
    0: "Harmful Content",
    1: "Non-Harmful Content",
}

In [ ]:
# Load the Maltese Semantic Analysis Dataset
dataset = load_dataset("DGurgurov/maltese_sa")

# Load different datasets into Data Frame variables
train_df = pd.DataFrame(dataset["train"])
val_df = pd.DataFrame(dataset["validation"])
test_df = pd.DataFrame(dataset["test"])

print("Train label counts:")
print(train_df["label"].value_counts())

print("\nValidation label counts:")
print(val_df["label"].value_counts())

print("\nTest label counts:")
print(test_df["label"].value_counts())

In [ ]:
# Metrics Function
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

In [ ]:
def get_run_dirs(run_name):
    run_dir = os.path.join(output_dir, run_name)
    return {
        "run_dir": run_dir,
        "results_dir": os.path.join(run_dir, "results"),
        "logs_dir": os.path.join(run_dir, "logs"),
        "saved_model_dir": os.path.join(run_dir, "saved_model"),
        "plots_dir": os.path.join(run_dir, "plots"),
        "predictions_dir": os.path.join(run_dir, "predictions"),
    }

In [ ]:
# Train the models
all_results = []

for run in model_runs:
    run_name = run["run_name"]
    max_length = run["max_length"]

    print("\n" + "=" * 80)
    print(f"Starting run: {run_name}")
    print("=" * 80)

    dirs = get_run_dirs(run_name)
    for path in dirs.values():
        os.makedirs(path, exist_ok = True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels = 2
    )

    def tokenize_function(batch):
        return tokenizer(batch["text"], truncation = True, max_length = max_length)

    tokenized_dataset = dataset.map(tokenize_function, batched = True)
    data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

    training_args = TrainingArguments(
        output_dir = dirs["results_dir"],
        eval_strategy = "epoch",
        save_strategy = "epoch",
        learning_rate = run["learning_rate"],
        per_device_train_batch_size = run["per_device_train_batch_size"],
        per_device_eval_batch_size = run["per_device_eval_batch_size"],
        num_train_epochs = run["num_train_epochs"],
        weight_decay = run["weight_decay"],
        logging_dir = dirs["logs_dir"],
        load_best_model_at_end = True,
        metric_for_best_model = "macro_f1",
        greater_is_better = True,
        disable_tqdm = True,
        save_total_limit = 1,
        report_to = "none",
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
    )

    trainer = Trainer(
        model = model,
        args = training_args,
        train_dataset = tokenized_dataset["train"],
        eval_dataset = tokenized_dataset["validation"],
        data_collator = data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=20)],
    )

    trainer.train()

    val_results = trainer.evaluate(tokenized_dataset["validation"])
    test_results = trainer.evaluate(tokenized_dataset["test"])

    print("Validation results:", val_results)
    print("Test results:", test_results)

    # Save best model + tokenizer
    trainer.save_model(dirs["saved_model_dir"])
    tokenizer.save_pretrained(dirs["saved_model_dir"])
    print(f"Saved model and tokenizer to: {dirs['saved_model_dir']}")

    # Save raw predictions for plotting in later cells
    predictions = trainer.predict(tokenized_dataset["test"])
    y_true = predictions.label_ids
    y_pred = np.argmax(predictions.predictions, axis=1)

    np.save(os.path.join(dirs["predictions_dir"], "y_true.npy"), y_true)
    np.save(os.path.join(dirs["predictions_dir"], "y_pred.npy"), y_pred)

    # Save run summary
    run_summary = {
        "run_name": run_name,
        "learning_rate": run["learning_rate"],
        "train_batch_size": run["per_device_train_batch_size"],
        "eval_batch_size": run["per_device_eval_batch_size"],
        "num_train_epochs": run["num_train_epochs"],
        "weight_decay": run["weight_decay"],
        "max_length": run["max_length"],
        "val_macro_accuracy": val_results.get("eval_macro_accuracy"),
        "val_macro_precision": val_results.get("eval_macro_precision"),
        "val_macro_recall": val_results.get("eval_macro_recall"),
        "val_f1": val_results.get("eval_f1"),
        "test_accuracy": test_results.get("eval_accuracy"),
        "test_macro_precision": test_results.get("eval_macro_precision"),
        "test_macro_recall": test_results.get("eval_macro_recall"),
        "test_macro_f1": test_results.get("eval_macro_f1"),
        "model_path": dirs["saved_model_dir"],
        "plots_path": dirs["plots_dir"],
        "results_dir": dirs["results_dir"],
    }

    with open(os.path.join(dirs["run_dir"], "run_summary.json"), "w", encoding="utf-8") as f:
        json.dump(run_summary, f, indent=4)

    all_results.append(run_summary)

results_df = pd.DataFrame(all_results).sort_values(by="val_macro_f1", ascending=False)
results_df.to_csv(os.path.join(output_dir, "all_model_results.csv"), index=False)

print("\nFinal comparison table:")
print(results_df)

In [ ]:
results_df = pd.DataFrame(all_results).sort_values(by="val_f1", ascending=False)
results_df.to_csv(os.path.join(output_dir, "all_model_results.csv"), index=False)

print("\nFinal comparison table:")
print(results_df)

In [ ]:
# Confusion Matrix Plot
for run in model_runs:
    run_name = run["run_name"]
    dirs = get_run_dirs(run_name)

    y_true_path = os.path.join(dirs["predictions_dir"], "y_true.npy")
    y_pred_path = os.path.join(dirs["predictions_dir"], "y_pred.npy")

    if not (os.path.exists(y_true_path) and os.path.exists(y_pred_path)):
        print(f"Skipping {run_name}: prediction files not found.")
        continue

    y_true = np.load(y_true_path)
    y_pred = np.load(y_pred_path)

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Harmful", "Non-Harmful"]
    )
    disp.plot(cmap="Blues", values_format="d", ax=ax)
    ax.set_title(f"Confusion Matrix - {run_name}")

    cm_path = os.path.join(dirs["plots_dir"], f"{run_name}_confusion_matrix.png")
    fig.savefig(cm_path, bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)